# 04 · Rung 2 — meaning, via fenic  ·  *optional coda*

Rungs 0–1 gave us **structure** (symbols, types, edges) — deterministically, no model. **Rung 2 is
*meaning*:** fold in the prose a human wrote (docstrings/comments) to capture *intent* the structure
can't express, and attach it to the **same graph nodes**.

Two architectural points this shows — the reason [fenic](https://github.com/typedef-ai/fenic) exists:

1. **Read the comments for intent**, not just the structure.
2. **Decompose.** Many small, focused ops — each fed *deterministic* input — not one mega-prompt. Small,
   grounded, auditable.

> Needs one `OPENROUTER_API_KEY` to run live (any model). With no key it renders **example** output so you
> can read the shape. The LLM step here is tiny (~8 `posting_list` nodes on a cheap model).\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bcallender/agent-context-workshop/blob/main/notebooks/04_rung2_fenic_coda.ipynb)  ·  *renders an example without a key; set OPENROUTER_API_KEY to run fenic live.*

In [1]:
# Colab bootstrap — no-op locally. In Colab: clone the repo, install it, hydrate the data.
import sys, os
if "google.colab" in sys.modules:
    import subprocess
    REPO = "/content/agent-context-workshop"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/bcallender/agent-context-workshop.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    subprocess.run(["bash", "scripts/setup_data.sh"], check=True)
    print("Colab ready — cloned, installed, data hydrated.")

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from context_workshop.parsers import load_rust_rustdoc
from context_workshop.parsers.schema import to_arrow

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOC = ROOT / "data/cache/cargo-target/doc"
CACHE = ROOT / "data/cache"
# fenic eagerly inits its client at Session creation, so the key must be in env first.
load_dotenv(ROOT / ".env")
HAS_KEY = bool(os.getenv("OPENROUTER_API_KEY"))

# Rung-1 rows we already built deterministically — one small crate, items that carry a docstring.
rd = [s for s in load_rust_rustdoc(DOC / "posting_list.json")
      if s.kind in ("struct", "trait", "function") and s.docstring][:8]
print(f"loaded {len(rd)} Rung-1 posting_list symbols with docstrings | OPENROUTER_API_KEY set: {HAS_KEY}")

loaded 8 Rung-1 posting_list symbols with docstrings | OPENROUTER_API_KEY set: False


## 1 · Read the prose for *intent*

A focused op over the **docstring** (the comment a human wrote) — one deterministic input, one question.
Not a mega-prompt that also asks five other things.

In [3]:
PROMPT_V1 = "In one concise sentence, what is this Rust item for?\n\n{{b}}"

# Example output shown when no key is set, so the notebook reads without OpenRouter.
EXAMPLE = [
    {"qualified_name": "posting_list::posting_list::PostingList",
     "summary": "A compressed inverted-index posting list mapping element ids to weights for one sparse dimension.",
     "stability": "STABLE"},
    {"qualified_name": "posting_list::view::PostingListView",
     "summary": "A non-owning, zero-copy view over a PostingList's encoded bytes.",
     "stability": "STABLE"},
    {"qualified_name": "posting_list::builder::PostingListBuilder",
     "summary": "Accumulates (id, weight) entries and finalizes them into a compressed PostingList.",
     "stability": "INTERNAL"},
]

if HAS_KEY:
    import fenic as fc
    sem = fc.SemanticConfig(
        language_models={"mini": fc.OpenRouterLanguageModel(model_name="openai/gpt-4o-mini", rpm=500, tpm=200_000)},
        default_language_model="mini")
    sess = fc.Session.get_or_create(
        fc.SessionConfig(app_name="rung2_demo", semantic=sem, db_path=str(CACHE / "fenic")))
    df = (sess.create_dataframe(to_arrow(rd))
          .with_column("blurb", fc.text.jinja("Rust {{k}} `{{qn}}`\n{{d}}", strict=False,
              k=fc.col("kind"), qn=fc.col("qualified_name"), d=fc.col("docstring")))
          .with_column("summary", fc.semantic.map(PROMPT_V1, b=fc.col("blurb"))))
    print("running fenic semantic.map (intent from docstrings)…")
else:
    df = None
    print("No OPENROUTER_API_KEY — showing EXAMPLE enriched rows below (set the key to run fenic live).")

No OPENROUTER_API_KEY — showing EXAMPLE enriched rows below (set the key to run fenic live).


## 2 · Decompose, don't interrogate

Stability is a **separate** small op — *not* another clause bolted onto the summary prompt. Want it to
weigh more? Feed *this* focused question **more deterministic input** (the signature + visibility). That
decomposition is how you get auditable semantic pipelines instead of one opaque mega-call.

In [4]:
STABILITY = ("Answer with exactly one word, STABLE or INTERNAL: is this a stable public API or an "
             "internal implementation detail?\n\nsignature: {{sig}}\nvisibility: {{vis}}")

if HAS_KEY:
    df = df.with_column("stability", fc.semantic.map(STABILITY, sig=fc.col("signature"), vis=fc.col("visibility")))
    rows = df.select("qualified_name", "summary", "stability").to_pylist()
else:
    rows = EXAMPLE

for r in rows[:3]:
    print(f"{r['qualified_name'].split('::')[-1]}\n  intent:    {r['summary']}\n  stability: {r['stability']}\n")

PostingList
  intent:    A compressed inverted-index posting list mapping element ids to weights for one sparse dimension.
  stability: STABLE

PostingListView
  intent:    A non-owning, zero-copy view over a PostingList's encoded bytes.
  stability: STABLE

PostingListBuilder
  intent:    Accumulates (id, weight) entries and finalizes them into a compressed PostingList.
  stability: INTERNAL



## 3 · Attach meaning to the **same** nodes

The enriched fields land on the existing graph nodes — grounded on the **same `file:line`**. Now the
agent's `get_symbol`/`search` surface *meaning alongside structure*: one node, two rungs.

In [5]:
if HAS_KEY:
    try:
        from context_workshop.graph import connect
        drv = connect(os.getenv("NEO4J_URI", "neo4j://localhost:7687"), os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "workshop123"))
        with drv.session() as s:
            for r in rows:
                s.run("MATCH (n:Symbol {qualified_name:$qn}) "
                      "SET n.summary=$s, n.stability=$st, n.has_meaning=true",
                      qn=r["qualified_name"], s=r["summary"], st=r["stability"])
            shown = s.run("MATCH (n:Symbol) WHERE n.has_meaning "
                          "RETURN n.qualified_name AS qn, n.filepath AS f, n.line_start AS l, n.summary AS s "
                          "LIMIT 3").data()
        drv.close()
        print("--- the graph now carries Rung-2 meaning, grounded on the SAME file:line ---")
        for r in shown:
            print(f"  {r['qn']} ({r['f']}:{r['l']})\n    → {r['s']}")
    except Exception as e:
        print("Neo4j not available — skipping graph attach (run `docker compose up -d`).", str(e)[:80])
else:
    print("(no key — example rows above are illustrative; with a key, these attach to graph nodes by file:line)")

(no key — example rows above are illustrative; with a key, these attach to graph nodes by file:line)
